# 02 · Estrategias, costes y métricas

**Objetivo:** comparar las estrategias frente a buy-and-hold, con y sin costes, y entender qué se come los beneficios.

**Necesitas:** los TODO 1.x, 2.x (motor), 3.x (estrategias) y 4.x (métricas). Checkpoint: `python -m pytest tests/test_backtest.py tests/test_strategies.py tests/test_metrics.py`.

**Teoría:** `docs/02_Motor_de_backtesting.pdf`, `docs/03_Estrategias.pdf` y `docs/04_Metricas.pdf`.

**Cómo usar este notebook.** Ejecuta las celdas en orden. Una celda que llama a una función que todavía es un TODO falla con `NotImplementedError: TODO x.y`: completa ese TODO, pasa su checkpoint de `pytest` y vuelve a ejecutarla (con `autoreload` no hace falta reiniciar el kernel).

Las celdas marcadas **`# TODO (análisis)`** son tuyas y no las corrige ningún test: sirven para mirar los datos y responder a las preguntas. Hasta que las completes no hacen nada (`...`).

In [ ]:
# Configuración: localiza la raíz del repo y recarga src/ al guardar cambios (no hace falta reiniciar)
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import backtest, config, data, metrics, plotting, strategies, walkforward

plotting.apply_style()
pd.set_option("display.width", 140)

In [ ]:
prices = data.load_prices()
params = config.STRATEGY_DEFAULTS
params

## 1. Señales

Cada estrategia devuelve una tabla fechas × activos con −1, 0 o 1. Antes de backtestear, mira cuánto tiempo pasa cada una invertida.

In [ ]:
signals = {name: strategies.get_strategy(name)(prices, **kw) for name, kw in params.items()}
invested = pd.DataFrame({name: (s != 0).mean() for name, s in signals.items()})
invested.style.format("{:.0%}")

In [ ]:
# El cruce de medias en el SPY durante los últimos 3 años (sombreado = invertido)
last = prices.index[-1] - pd.DateOffset(years=3)
close = prices["SPY"]
fast = close.rolling(params["sma_crossover"]["fast"]).mean()
slow = close.rolling(params["sma_crossover"]["slow"]).mean()
signal = signals["sma_crossover"]["SPY"]

fig, ax = plt.subplots(figsize=(10, 4))
window = close.index >= last
ax.fill_between(close.index[window], 0, 1, where=signal[window] > 0, transform=ax.get_xaxis_transform(),
                color=plotting.NEUTRAL_FILL, label="señal = 1")
ax.plot(close[window], color=plotting.PALETTE[0], label="SPY")
ax.plot(fast[window], color=plotting.PALETTE[1], label="media 50")
ax.plot(slow[window], color=plotting.PALETTE[2], label="media 200")
ax.set_title("Cruce de medias 50/200 en el SPY")
ax.legend(loc="upper left");

In [ ]:
# Bandas de Bollinger en el SPY durante el último año, con la señal (estrategia con estado)
last = prices.index[-1] - pd.DateOffset(years=1)
kw = params["bollinger_mean_reversion"]
mid = close.rolling(kw["window"]).mean()
band = kw["n_std"] * close.rolling(kw["window"]).std()
signal = signals["bollinger_mean_reversion"]["SPY"]

fig, ax = plt.subplots(figsize=(10, 4))
window = close.index >= last
ax.fill_between(close.index[window], 0, 1, where=signal[window] > 0, transform=ax.get_xaxis_transform(),
                color=plotting.NEUTRAL_FILL, label="señal = 1")
ax.plot(close[window], color=plotting.PALETTE[0], label="SPY")
ax.plot(mid[window], color=plotting.PALETTE[1], label="media 20")
ax.plot((mid - band)[window], color=plotting.PALETTE[2], label="banda inferior")
ax.set_title("Bollinger (20, 2σ) en el SPY: se entra bajo la banda y se sale en la media")
ax.legend(loc="upper left");

## 2. Backtests con y sin costes

In [ ]:
results = {name: backtest.run_backtest(prices, s) for name, s in signals.items()}
gross = {name: backtest.run_backtest(prices, s, commission_bps=0, slippage_bps=0) for name, s in signals.items()}
results["sma_crossover"]

## 3. Tabla comparativa

In [ ]:
table_net = metrics.summary_table(results)
table_gross = metrics.summary_table(gross)
print("Con costes (5 + 5 bps)")
display(metrics.format_summary(table_net))
print("Sin costes")
display(metrics.format_summary(table_gross))

El README pide verificar el Sharpe contra `quantstats`. Las diferencias deberían ser prácticamente 0 (el test exige menos de 1e-10):

In [ ]:
metrics.compare_with_quantstats(results["sma_crossover"])

In [ ]:
# TODO (análisis): ¿cuadran los costes?
# Coste anual aproximado ≈ turnover anual × (comisión + slippage) / 10 000.
# Compáralo, por estrategia, con la diferencia CAGR sin costes − CAGR con costes.
# Pista: las columnas "Turnover anual" y "CAGR" de table_net y table_gross.
cost_check = ...
cost_check

## 4. Equity y drawdown

In [ ]:
plotting.plot_equity_and_drawdown(results);

In [ ]:
for name, res in results.items():
    print(f"{name:26} {metrics.max_drawdown(res)}")

## 5. Impacto de los costes

Las estrategias de reversión a la media operan mucho. ¿Cuánto coste aguantan?

In [ ]:
# TODO (análisis): Sharpe de cada estrategia con costes totales de 0, 5, 10, 20, 30 y 50 bps.
# Reparte cada coste a partes iguales entre comisión y slippage (solo importa la suma).
# Resultado: DataFrame con índice = bps (llámalo "bps") y una columna por estrategia.
costs_bps = [0, 5, 10, 20, 30, 50]
sensitivity = ...
sensitivity

In [ ]:
# Se dibuja cuando hayas completado la celda anterior
if isinstance(sensitivity, pd.DataFrame):
    plotting.plot_cost_sensitivity(sensitivity)

> **Preguntas.** ¿A partir de cuántos bps deja de compensar Bollinger frente a buy-and-hold? ¿Qué columna de la tabla comparativa lo predecía? ¿Son realistas 5 + 5 bps para un ETF líquido? ¿Y para una acción pequeña?

## 6. Variantes (opcional)

In [ ]:
# TODO (análisis): compara variantes de las estrategias con summary_table + format_summary:
#   - momentum 12-0 frente a 12-1 (skip_months=1)
#   - cruce de medias solo largo frente a largo/corto (allow_short=True)
# ¿Mejora alguna variante de forma clara, o las diferencias son pequeñas comparadas con el ruido?
variants = ...

## 7. Extensión: volatility targeting (TODO 6.1)

`run_backtest(..., vol_target=0.10)` escala la posición de cada activo para apuntar a un 10 % de volatilidad anual. Teoría: `docs/06_Volatility_targeting.pdf`.

In [ ]:
# TODO (análisis): compara buy_and_hold y sma_crossover con y sin vol_target=0.10.
# ¿Qué cambia más: el CAGR, la volatilidad, el drawdown o el Sharpe? ¿Por qué?
vol_targeted = ...

## 8. Preguntas de reflexión

1. ¿Alguna estrategia bate a buy-and-hold en CAGR? ¿Y en Sharpe o en drawdown? ¿Qué preferirías y por qué?
2. La cartera es equiponderada entre 10 activos muy distintos (TLT, GLD, AAPL...). ¿Cómo afecta eso a la comparación?
3. Estos parámetros (50/200, 20 días y 2σ) son los "de libro", no los optimizados. ¿Por qué eso hace que esta tabla sea más creíble que una con parámetros ajustados? (Lo verás en el notebook 03.)
4. ¿Qué limitaciones del README afectan más a cada estrategia? Piensa en los cortos sin coste de préstamo y en el slippage fijo.